# Gold Layer

In [0]:
%sql
CREATE OR REPLACE TABLE gold_distric_operator_summary AS
SELECT 
    district_name,
    operator_name,
    COUNT(session_id) AS total_sessions,
    ROUND(SUM(kwh_charged), 2) AS total_kwh,
    ROUND(AVG(cost_per_kwh), 2) AS avg_price_per_kwh,
    ROUND(SUM(total_cost), 2) AS total_revenue_pln
FROM silver_charging_sessions
GROUP BY district_name, operator_name
ORDER BY total_kwh DESC;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_hourly_grid_demand AS
SELECT 
    HOUR(session_start_time) AS hour_num,
    
    concat(
        lpad(HOUR(session_start_time), 2, '0'), ':00'
    ) AS time_window,

    CASE 
        WHEN HOUR(session_start_time) BETWEEN 7 AND 10 THEN 'Morning summit'
        WHEN HOUR(session_start_time) BETWEEN 17 AND 21 THEN 'Evening summit'
        WHEN HOUR(session_start_time) >= 22 OR HOUR(session_start_time) <= 5 THEN 'Night time plunge'
        ELSE 'Other'
    END AS grid_load_zone,

    COUNT(session_id) AS total_sessions,
    ROUND(SUM(kwh_charged) / 1000, 2) AS total_mwh_delivered,
    ROUND(AVG(kwh_charged), 2) AS avg_kwh_per_session,
    ROUND(SUM(total_cost), 2) AS total_revenue_pln

FROM silver_charging_sessions
GROUP BY HOUR(session_start_time)
ORDER BY hour_num;

# Silver Layer

In [0]:
%sql
CREATE OR REPLACE TABLE silver_charging_sessions AS
SELECT 
    s.session_id,
    s.customer_id,
    s.station_id,
    s.session_start_time,
    
    -- Zaokrąglenia
    ROUND(s.kwh_charged, 2) AS kwh_charged,
    ROUND(s.cost_per_kwh, 2) AS cost_per_kwh,
    ROUND(s.total_cost, 2) AS total_cost,
    
    -- Informacje o samochodach
    c.car_model,
    c.battery_capacity_kwh,
    
    -- Informacje o stacji 
    st.operator_name,
    st.district_name,
    ROUND(st.latitude, 4) AS latitude,
    ROUND(st.longitude, 4) AS longitude,
    
    current_timestamp() AS _silver_processed

FROM bronze_charging_sessions s
JOIN bronze_customers c ON s.customer_id = c.customer_id
JOIN bronze_charging_stations st ON s.station_id = st.station_id
WHERE 
    
    s.session_id IS NOT NULL 
    AND s.customer_id IS NOT NULL 
    AND s.station_id IS NOT NULL
    
    AND s.kwh_charged > 0
    AND s.total_cost > 0
    
    AND s.session_start_time <= current_timestamp()
    
    AND s.kwh_charged <= c.battery_capacity_kwh
   
    AND ABS(s.total_cost - (s.kwh_charged * s.cost_per_kwh)) < 0.01
    
    -- Must be in Warsaw
    AND st.latitude BETWEEN 52.09 AND 52.37
    AND st.longitude BETWEEN 20.85 AND 21.27;

# Automated loading of Bronze Layer

In [0]:
from pyspark.sql.functions import current_timestamp, col

base_path = "/Workspace/Users/szyrydz@gmail.com/Warsaw EV Charging/"

tables_to_load = {
    "customers.csv": "bronze_customers",
    "charging_stations.csv": "bronze_charging_stations",
    "districts.csv": "bronze_districts",
    "charging_sessions.csv": "bronze_charging_sessions"
}

for file_name, table_name in tables_to_load.items():
    full_path = base_path + file_name
    
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(full_path) \
        .withColumn("_timestamp", current_timestamp()) \
        .withColumn("_source_file", col("_metadata.file_path"))
        
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)
